In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import yaml
from core.helpers import get_config_file, get_data_out, get_data_computed_in, UniqueKeyLoader
from sm.model_manipulation import TimeSeriesScaler, generate_initial_model, EnergySystemDesignConverter
from sm.sampling import SampleGenerator
from sm.routines import (generate_aggregated_input_data, design_optimization_var_ts,
                          operational_optimization_const_ts, operational_optimization_var_ts)

PROJECT_FILE = 'project.yaml'
CASE = 'JP1'
ROOT = os.path.dirname(os.path.dirname(os.getcwd()))
DATA_IN = os.path.join(ROOT, 'data', CASE, 'Input_Data')
CONFIG = get_config_file(DATA_IN, PROJECT_FILE)

DATA_COMPUTED_IN = get_data_computed_in(DATA_IN, PROJECT_FILE)
DATA_OUT         = get_data_out(DATA_IN, PROJECT_FILE)

### README:
In this notebook, the usage of the SampleGenerator and the subsequent data_generation (manipulating a pyomo model directly)
1. Demonstration of sampling different **timeseries**
2. Demonstration of sampling different **design capacities**
3. Demonstration of sampling different **timeseries and design capacities**.


### 1. Demonstration of sampling different **timeseries**

Here, design capacities are not sampled. Two use cases
* **Design Surrogate** for different conditions
    * Input: timeseries, Outputs: process_capacites and KPIs (OPEX, ...)
    * Here, Design optimizations are required to generate the training data
    * The following Section focuses on the Design Surrogate Application
* **Operational Surrogate** trained for **one specific design** and different conditions
    * Input: timeseries, Outputs: KPIs (OPEX, ...)
    * Here, Operational optimizations are required to generate the training data
    * Currently not used, therefore not shown in this notebook


In [ ]:
# Initialize SampleGenerator

sample_generator = SampleGenerator(
    DATA_IN, PROJECT_FILE,
    sample_processes=False, # <- IMPORANT
    sample_timeseries=True # <- IMPORANT
)

# Per default, all timeseries mentioned in uncertainties.yaml will be sampled.
# If you want to ignore one timeseries, you can either:
# * change uncertainties.yaml. This is intended for more fundamental changes. (Not shown in this notebook)
# * use sample_generator.set_timeseries_ignore_list . Indended use is for comparing different configurations automatically in one script.

# Print ignored timeseries "Before" (which is empty, since it has not been set yet)
print(f"timeseries_ignored:\n\tBefore: {sample_generator.timeseries_ignored}")

# Set ignored timeseries. The names are the keys in uncertainties.yaml
sample_generator.set_timeseries_ignore_list([
    # 'ts_avail_pv', 'ts_demand_heat', 'ts_pricebuy_gas'
    ])

# Print ignored timeseries "After":
print(f"\tAfter:  {sample_generator.timeseries_ignored}")

# Report min, max and mean values for extreme ts vertex points:
sample_generator.report_extreme_ts(report_aggregated_timeseries=True)

In [ ]:
# Use sample_generator to generate lhs_sample:
lhs_sample_df = sample_generator.sample_lhs(
    n_points     = 10000,  # Number of points
    seed         = 0,
    resample     = True, # If True, samples with 'peak' < 'mean' are resampled (indended configuration).
    add_vertices = True,# If True, all vertices of the input hyperspace are added.
)

print(f"lhs_sample_df.shape: {lhs_sample_df.shape}")

In [ ]:
# Preperation. In a routine over several samples, this part would only be executed once (at the beginning).
# Aggregate input data
agg_dict = generate_aggregated_input_data(DATA_IN, PROJECT_FILE)

# Initialize the optimizer and a blank_model
optimizer, blank_model = generate_initial_model(DATA_IN, PROJECT_FILE, 'design')

# Initialize the timeseries scaler
ts_scaler = TimeSeriesScaler(
    DATA_IN,
    PROJECT_FILE,
    blank_model=blank_model,
    timeseries_ignored=sample_generator.timeseries_ignored
)


In [ ]:
result_df = design_optimization_var_ts(
    df_in=lhs_sample_df,
    data_in_path=DATA_IN,
    project_file=PROJECT_FILE,
    timeseries_ignored=sample_generator.timeseries_ignored,
    debug=True # True: Print solver log, for each optimization: export output_dict and result_plot
)

result_df

In [ ]:
result_df.to_csv(os.path.join(DATA_OUT, 'design_10800.csv'), index=False)

In [ ]:
# Alternative to solving in this notebook: Use HPC. 
# Therefore, export lhs_sample_df and timeseries_ignored
lhs_sample_df.to_csv(os.path.join(DATA_COMPUTED_IN, f'design_opt_var_ts_lhs_sample_{len(lhs_sample_df)}.csv'), index=False)

# Export timeseries ignored
with open(os.path.join(DATA_COMPUTED_IN, 'design_opt_var_ts_ts_ignore.yaml'), 'w') as f:
    yaml.dump(sample_generator.timeseries_ignored, f)

### 2. Demonstration of sampling different **design capacities**

Here, timeseries are not sampled. One use-case:
* **Operational Surrogate** trained for different designs but **without different conditions**
    * Input: design capacities, Outputs: KPIs (OPEX, ...)
    * Here, Operational optimizations are required to generate the training data


In [ ]:
# Initialize SampleGenerator

sample_generator = SampleGenerator(
    DATA_IN, PROJECT_FILE,
    sample_processes=True, # <- IMPORANT
    sample_timeseries=False # <- IMPORANT
)

####### FIX TECHNOLOGIES ###########
# Per default, all technologies from technologies.yaml will be sampled.
# If you want to change (fix) a technology, you can either:
# * change technologies.yaml. This is intended for more fundamental changes. (Not shown in this notebook)
# * use sample_generator.processes_fixed . Indended use is for comparing different configurations automatically in one script.

# Print fixed timeseries "Before" (which is empty, since it has not been set yet)
print(f"processes_fixed:\n\tBefore: {sample_generator.processes_fixed}")

# Set fixed processes. The keys correspond to the keys in technologies.yaml
sample_generator.set_processes_fixed( {'CHP': 600, 'HTHP': 0, 'battery': 1000, 'AC': 200, 'CC':500} )

# Print fixed processes "After":
print(f"\tAfter:  {sample_generator.processes_fixed}")

####### CHANGE TECHNOLOGY BOUNDS ###########
# Per default, all bounds are taken from technologies.yaml.
# If you want to change the bounds , you can either:
# * change technologies.yaml. This is intended for more fundamental changes. (Not shown in this notebook)
# * use sample_generator.set_processes_custom_bounds. Indended use is for comparing different configurations automatically in one script.

# Print processes_custom_bounds "Before" (which is empty, since it has not been set yet)
print(f"\nprocesses_custom_bounds:\n\tBefore: {sample_generator.processes_custom_bounds}")

# Set custom bounds. The keys correspond to the keys in technologies.yaml
sample_generator.set_processes_custom_bounds(
    {
        'max': {'B':9000, 'TES': 14000},
        'min': {'B':0}
    }
)

# Print processes_custom_bounds "After":
print(f"\tAfter:  {sample_generator.processes_custom_bounds}")

# Report min, max and mean values for extreme ts vertex points:
sample_generator.report_extreme_ts(report_aggregated_timeseries=False)

In [ ]:
# Use sample_generator to generate lhs_sample:
lhs_sample_df = sample_generator.sample_lhs(
    n_points     = 1, #int(1e6),  # Number of points
    seed         = 0,
    resample     = True, # If True, samples with 'peak' < 'mean' are resampled (indended configuration).
    add_vertices = True,# If True, all vertices of the input hyperspace are added.
)
print(f"lhs_sample_df.shape: {lhs_sample_df.shape}")

In [ ]:
# Preperation. In a routine over several samples, this part would only be executed once (at the beginning).
# Aggregate input data
generate_aggregated_input_data(DATA_IN, PROJECT_FILE)

# Initialize the optimizer and a blank_model
optimizer, blank_model = generate_initial_model(
    DATA_IN,
    PROJECT_FILE,
    'operation'   # <- important
) 

# Initialize the Energy System Design Converter:
esd_converter = EnergySystemDesignConverter(DATA_IN, PROJECT_FILE)

In [ ]:
result_df = operational_optimization_const_ts(
    df_in = lhs_sample_df[:5],
    data_in_path=DATA_IN,
    project_file=PROJECT_FILE,
    debug=True # True: Print solver log, for each optimization: export output_dict and result_plot
)

In [ ]:
result_df

In [ ]:
# Alternative to solving in this notebook: Use HPC. 
# Therefore, export lhs_sample_df and timeseries_ignored
lhs_sample_df.to_csv(os.path.join(DATA_COMPUTED_IN, f'operational_opt_const_ts_lhs_sample_{len(lhs_sample_df)}.csv'), index=False)


### 3. Demonstration of sampling different **timeseries and design capacities**

Here, design capacities and timeseries are sampled. One use-case:
* **Operational Surrogate** trained for different designs and different conditions
    * Input: timeseries, design capacities, Outputs: KPIs (OPEX, ...)
    * Here, Operational optimizations are required to generate the training data


In [ ]:
# Initialize SampleGenerator

sample_generator = SampleGenerator(
    DATA_IN, PROJECT_FILE,
    sample_processes=True, # <- IMPORANT
    sample_timeseries=True # <- IMPORANT
)

print(f"timeseries_ignored:\n\tBefore: {sample_generator.timeseries_ignored}")
sample_generator.set_timeseries_ignore_list(
    [
        # 'ts_avail_pv',
        # 'ts_demand_cold'
        ]
    )
print(f"\tAfter:{sample_generator.timeseries_ignored}")

# sample_generator.set_processes_fixed({'CHP': 6000})

# Report min, max and mean values for extreme ts vertex points:
sample_generator.report_extreme_ts(report_aggregated_timeseries=True)

In [ ]:

lhs_sample_df = sample_generator.sample_lhs(
    n_points=100000,
    seed=2,
    resample=True,
    add_vertices=False,
)

print(f"lhs_sample_df.shape: {lhs_sample_df.shape}")

In [ ]:
lhs_sample_df

In [ ]:
result_df = operational_optimization_var_ts(
    df_in = lhs_sample_df,
    data_in_path=DATA_IN,
    optimization_type='operation', #'max_overcapacity',
    project_file=PROJECT_FILE,
    timeseries_ignored= sample_generator.timeseries_ignored,
    debug=False # True: Print solver log, for each optimization: export output_dict and result_plot
)
result_df_max_oc = operational_optimization_var_ts(
    df_in = lhs_sample_df,
    data_in_path=DATA_IN,
    optimization_type='max_overcapacity',
    project_file=PROJECT_FILE,
    timeseries_ignored= sample_generator.timeseries_ignored,
    debug=False # True: Print solver log, for each optimization: export output_dict and result_plot
)

In [ ]:
result_df.to_csv(os.path.join(DATA_OUT, 'operation_100000.csv'), index=False)
result_df_max_oc.to_csv(os.path.join(DATA_OUT, 'op_var_max_overcapacity_100000.csv'), index=False)


In [ ]:
# Alternative to solving in this notebook: Use HPC. 
# Therefore, export lhs_sample_df and timeseries_ignored
lhs_sample_df.to_csv(os.path.join(DATA_COMPUTED_IN, f'operational_opt_var_ts_lhs_sample_{len(lhs_sample_df)}.csv'), index=False)

# Export timeseries ignored
with open(os.path.join(DATA_COMPUTED_IN, 'operational_opt_var_ts_ts_ignore.yaml'), 'w') as f:
    yaml.dump(sample_generator.timeseries_ignored, f)

# Analyze samples

### 1. Design Optimization of different timeseries

In [ ]:
output_df_design = pd.read_csv(os.path.join(DATA_OUT,'design_test_results.csv'))

In [ ]:
# generate heatmap of pearson correlation from output_df_design
import seaborn as sns
plt.close('all')
fig, ax = plt.subplots(1,1, figsize=(15,9))

# filter columns, such that lost_load is not included
output_df_design = output_df_design.loc[:, ~output_df_design.columns.str.contains('lost_load')]
sns.heatmap(
    output_df_design.corr(),
    annot=True,
    ax=ax,
    fmt=".2f",
    mask=np.triu(output_df_design.corr()),
    cmap='coolwarm')
# use a color map that has white color for 0 correlation
cmap = sns.diverging_palette(220, 20, as_cmap=True)
plt.show()

# save fig with timestamp
# fig.savefig(os.path.join(DATA_OUT, f"correlation_heatmap_design_{time.strftime('%Y-%m-%d_%H.%M.%S', time.localtime())}.png"))


In [ ]:
# plot distribution of all columns in output_df_design
# (its 36 columns, plot them in 6x6 grid)
plt.close('all')
fig, axs = plt.subplots(6,6, figsize=(13,10))
# use small font size for everything
plt.rcParams.update({'font.size': 6})
for i, column in enumerate(output_df_design.columns):
    ax = axs[i//6, i%6]
    output_df_design[column].plot.hist(ax=ax, bins=50)
    ax.set_title(column)
    ax.grid()
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

plt.tight_layout()

# save fig with timestamp
fig.savefig(os.path.join(DATA_OUT, f"distribution_design_{time.strftime('%Y-%m-%d_%H.%M.%S', time.localtime())}.png"))

### 2. Operational Optimazation for different designs but constant timeseries

In [ ]:
output_df_operational_const = pd.read_csv(os.path.join(DATA_OUT,'operation_const_ts_TD5.csv'))

# calculate OPEX_without lost_load and TAC without lost_load

value_of_lost_load = 100 # EUR/kWh
output_df_operational_const['OPEX_without_ll'] = output_df_operational_const['OPEX'] - output_df_operational_const['lost_load'] * value_of_lost_load
output_df_operational_const['TAC_without_ll'] = output_df_operational_const['TAC'] - output_df_operational_const['lost_load'] * value_of_lost_load

In [ ]:
# generate heatmap of pearson correlation from output_df_operational_const
import seaborn as sns
plt.close('all')
fig, ax = plt.subplots(1,1, figsize=(15,9))
# change the annotation to 2 decimal places
# only generate lower triangle of the heatmap
sns.heatmap(
    output_df_operational_const.corr(),
    annot=True,
    ax=ax,
    fmt=".2f",
    mask=np.triu(output_df_operational_const.corr()),
    cmap='coolwarm')
# use a color map that has white color for 0 correlation
cmap = sns.diverging_palette(220, 20, as_cmap=True)
plt.show()

# save fig with timestamp
fig.savefig(os.path.join(DATA_OUT, f"correlation_heatmap_operation_const_{time.strftime('%Y-%m-%d_%H.%M.%S', time.localtime())}.png"))


In [ ]:
# plot distribution of all columns in output_df_operational_const
# (its 36 columns, plot them in 6x6 grid)
plt.close('all')
fig, axs = plt.subplots(6,6, figsize=(13,10))
# use small font size for everything
plt.rcParams.update({'font.size': 6})
for i, column in enumerate(output_df_operational_const.columns):
    ax = axs[i//6, i%6]
    output_df_operational_const[column].plot.hist(ax=ax, bins=50)
    ax.set_title(column)
    ax.grid()
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

plt.tight_layout()

# save fig with timestamp
fig.savefig(os.path.join(DATA_OUT, f"distribution_operation_const_{time.strftime('%Y-%m-%d_%H.%M.%S', time.localtime())}.png"))

### 3. Operational Optimazation for different timeseries and designs

In [ ]:
output_df_operation_var = pd.read_csv(os.path.join(DATA_OUT,'op_var_dmpi_200000.csv'))

# Sort by id:
output_df_operation_var = output_df_operation_var.sort_values(by='id')

# change id to int:
output_df_operation_var['id'] = output_df_operation_var['id'].astype(int)

# calculate OPEX_without lost_load and TAC without lost_load
value_of_lost_load = 100 # EUR/kWh
output_df_operation_var['OPEX_without_ll'] = output_df_operation_var['OPEX'] - output_df_operation_var['lost_load'] * value_of_lost_load
output_df_operation_var['TAC_without_ll'] = output_df_operation_var['TAC'] - output_df_operation_var['lost_load'] * value_of_lost_load

output_df_operation_var

In [ ]:
# count, how many entries of OPEX_without_ll are negative
n_negative_opex = (output_df_operation_var['OPEX_without_ll'] < 0).sum()
# print the absolute number and the percentage
print(f"Number of negative OPEX_without_ll: {n_negative_opex}")
print(f"Percentage of negative OPEX_without_ll: {n_negative_opex / len(output_df_operation_var) * 100:.2f}%")


# Count how many samples have lost load > 0
n_no_lost_lost = (output_df_operation_var['lost_load'] > 0).sum()


print(f"Number of samples with lost load: {n_no_lost_lost}")
print(f"Percentage of samples with lost load: {n_no_lost_lost / len(output_df_operation_var) * 100:.2f}%")

In [ ]:
# repeat interesting cacluations to save the output dict and plot the results:
lhs_sample_df_op_var = pd.read_csv(os.path.join(DATA_COMPUTED_IN,"operational_opt_var_ts_lhs_sample.csv"))

with open(os.path.join(DATA_COMPUTED_IN, 'operational_opt_var_ts_ts_ignore.yaml'), 'rb') as f:
    timeseries_ignored = yaml.load(f, Loader=UniqueKeyLoader)


In [ ]:
result_df = operational_optimization_var_ts(
    df_in = lhs_sample_df.loc[574975:574975],
    data_in_path=DATA_IN,
    project_file=PROJECT_FILE,
    timeseries_ignored= sample_generator.timeseries_ignored,
    debug=True # True: Print solver log, for each optimization: export output_dict and result_plot
)

In [ ]:
# generate heatmap of pearson correlation from output_df_operation_var
import seaborn as sns
plt.close('all')
fig, ax = plt.subplots(1,1, figsize=(15,9))
# change the annotation to 2 decimal places
# only generate lower triangle of the heatmap
sns.heatmap(
    output_df_operation_var.corr(),
    annot=True,
    ax=ax,
    fmt=".2f",
    mask=np.triu(output_df_operation_var.corr()),
    cmap='coolwarm',
    annot_kws={"size": 5})
# use a color map that has white color for 0 correlation
cmap = sns.diverging_palette(220, 20, as_cmap=True)
plt.show()

# save fig with timestamp
# fig.savefig(os.path.join(DATA_OUT, f"correlation_heatmap_{time.strftime('%Y-%m-%d_%H.%M.%S', time.localtime())}.png"))

In [ ]:
# plot distribution of all columns in output_df_operation_var
# (its 36 columns, plot them in 6x6 grid)
plt.close('all')
fig, axs = plt.subplots(6,6, figsize=(13,10))
# use small font size for everything
plt.rcParams.update({'font.size': 6})
for i, column in enumerate(output_df_operation_var.columns):
    ax = axs[i//6, i%6]
    output_df_operation_var[column].plot.hist(ax=ax, bins=50)
    ax.set_title(column)
    ax.grid()
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

plt.tight_layout()

# save fig with timestamp
fig.savefig(os.path.join(DATA_OUT, f"distribution_{time.strftime('%Y-%m-%d_%H.%M.%S', time.localtime())}.png"))